# Football Match Prediction - Model Training

Three machine learning models for English Premier League match prediction:

| Model | Prediction | Type |
|-------|-----------|------|
| **Match Result** | Home Win (H), Draw (D), or Away Win (A) | Multiclass |
| **Over/Under 2.5** | Total goals over or under 2.5 | Binary |
| **BTTS** | Both Teams To Score (Yes/No) | Binary |

**Approach:** GridSearchCV with Pipelines (scaler + classifier) over LogisticRegression, RandomForest, and GradientBoosting.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

---
## 1. Load the Data

In [ ]:
df = pd.read_csv('data/dataset_features.csv')
raw = pd.read_csv('data/E0_Cleaned.csv')

df['Date'] = pd.to_datetime(df['Date'])
raw['Date'] = pd.to_datetime(raw['Date'])

df = df.merge(
    raw[['HomeTeam', 'AwayTeam', 'Date', 'FTHG', 'FTAG']],
    on=['HomeTeam', 'AwayTeam', 'Date'],
    how='left'
)
df = df.dropna(subset=['FTHG', 'FTAG'])

print(f'Dataset: {df.shape[0]} matches, {df.shape[1]} columns')
df.head()

---
## 2. Explore the Data

Match result distribution, goals scored, and BTTS rates.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

result_map = {'H': 'Home Win', 'D': 'Draw', 'A': 'Away Win'}
df['Result'] = df['Target'].map(result_map)
colors = ['#2ecc71', '#f39c12', '#e74c3c']

df['Result'].value_counts().plot(kind='bar', ax=axes[0], color=colors, edgecolor='black')
axes[0].set_title('Match Result Distribution')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

df['TotalGoals'] = df['FTHG'] + df['FTAG']
axes[1].hist(df['TotalGoals'], bins=range(0, 10), color='steelblue', edgecolor='black', alpha=0.7)
axes[1].axvline(x=2.5, color='red', linestyle='--', label='2.5 line')
axes[1].set_title('Total Goals per Match')
axes[1].set_xlabel('Goals')
axes[1].legend()

btts = ((df['FTHG'] > 0) & (df['FTAG'] > 0)).value_counts()
btts.index = ['Both Score', 'Clean Sheet']
btts.plot(kind='pie', ax=axes[2], autopct='%1.1f%%', colors=['#3498db', '#e74c3c'],
          startangle=90, textprops={'fontsize': 11})
axes[2].set_title('Both Teams to Score')
axes[2].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
print('--- Dataset Summary ---')
print(f'Total matches: {len(df)}')
print(f'Date range: {df["Date"].min().date()} to {df["Date"].max().date()}')
print(f'Unique teams: {len(set(df["HomeTeam"].unique()) | set(df["AwayTeam"].unique()))}')
print(f'\nTarget distribution:')
print(df['Target'].value_counts().to_string())
print(f'\nFeature columns: {len([c for c in df.columns if c.startswith(("home_", "away_", "diff_", "h2h_"))])}')

---
## 3. Feature Distributions

In [ ]:
feature_cols = [
    'home_avg_goals', 'home_avg_shots', 'home_avg_conceded',
    'away_avg_goals', 'away_avg_shots', 'away_avg_conceded',
    'diff_avg_goals', 'diff_avg_shots', 'diff_avg_conceded'
]

fig, axes = plt.subplots(3, 3, figsize=(14, 12))
for i, col in enumerate(feature_cols):
    ax = axes[i // 3][i % 3]
    df[col].hist(bins=20, ax=ax, color='steelblue', edgecolor='black', alpha=0.7)
    ax.set_title(col.replace('_', ' ').title())
    ax.set_xlabel('')

plt.suptitle('Feature Distributions', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
corr = df[feature_cols + ['TotalGoals']].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

---
## 4. Prepare Features and Targets

In [ ]:
cols_to_drop = ['HomeTeam', 'AwayTeam', 'Date', 'Target', 'FTHG', 'FTAG', 'Result', 'TotalGoals']
X = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
y = df['Target']

print(f'Features: {X.shape[1]} columns, {X.shape[0]} rows')
print(f'Target classes: {sorted(y.unique())}')
print(f'\nFeature names:')
for i, col in enumerate(X.columns, 1):
    print(f'  {i:2d}. {col}')

---
## 5. Train-Test Split

80/20 stratified split.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape[0]} matches')
print(f'Test set:     {X_test.shape[0]} matches')
print(f'\nClass split in training set:')
print(y_train.value_counts().to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

y_train.value_counts().rename(result_map).plot(kind='bar', ax=axes[0], color=colors, edgecolor='black')
axes[0].set_title('Training Set Distribution')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

y_test.value_counts().rename(result_map).plot(kind='bar', ax=axes[1], color=colors, edgecolor='black')
axes[1].set_title('Test Set Distribution')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

---
## 6. Train with GridSearchCV

Compare LogisticRegression, RandomForest, and GradientBoosting with hyperparameter tuning.
Each model is wrapped in a Pipeline with StandardScaler.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

def optimize_models(X_train, y_train):
    pipe_lr = Pipeline([('scaler', StandardScaler()), ('classifier', LogisticRegression(max_iter=1000, random_state=42))])
    pipe_rf = Pipeline([('scaler', StandardScaler()), ('classifier', RandomForestClassifier(random_state=42))])
    pipe_gb = Pipeline([('scaler', StandardScaler()), ('classifier', GradientBoostingClassifier(random_state=42))])

    models = {
        'LogisticRegression': (pipe_lr, {
            'classifier__C': [0.1, 1, 10],
            'classifier__penalty': ['l2'],
            'classifier__class_weight': ['balanced', None]
        }),
        'RandomForest': (pipe_rf, {
            'classifier__n_estimators': [100, 200],
            'classifier__max_depth': [5, 10, None],
            'classifier__class_weight': ['balanced', None]
        }),
        'GradientBoosting': (pipe_gb, {
            'classifier__n_estimators': [100, 200],
            'classifier__learning_rate': [0.05, 0.1],
            'classifier__max_depth': [3, 5]
        })
    }

    results = {}
    for name, (pipeline, params) in models.items():
        print(f'\nTraining {name}...')
        grid = GridSearchCV(pipeline, params, cv=5, scoring='f1_macro', n_jobs=-1)
        grid.fit(X_train, y_train)
        results[name] = grid
        print(f'  Best params: {grid.best_params_}')
        print(f'  Best CV F1: {grid.best_score_:.4f}')

    best_name = max(results, key=lambda k: results[k].best_score_)
    print(f'\nBest model: {best_name} (F1={results[best_name].best_score_:.4f})')
    return results

In [ ]:
results = optimize_models(X_train, y_train)

---
## 7. Evaluate Match Result Model

In [ ]:
best_model = results['GradientBoosting'].best_estimator_
y_pred = best_model.predict(X_test)

print(f'Test Accuracy: {accuracy_score(y_test, y_pred):.2%}')
print()
print(classification_report(y_test, y_pred, target_names=['Away Win', 'Draw', 'Home Win']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test, y_pred, labels=['A', 'D', 'H'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Away Win', 'Draw', 'Home Win'],
            yticklabels=['Away Win', 'Draw', 'Home Win'], ax=axes[0])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Match Result - Confusion Matrix')

importances = pd.Series(best_model.named_steps['classifier'].feature_importances_, index=X.columns)
importances.nlargest(12).plot(kind='barh', ax=axes[1], color='steelblue', edgecolor='black')
axes[1].set_title('Match Result - Feature Importance')
axes[1].set_xlabel('Importance')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

---
## 8. Train Over/Under 2.5 Goals Model

In [ ]:
y_ou = (df['FTHG'] + df['FTAG'] > 2).astype(int)
print('Over/Under target distribution:')
print(y_ou.value_counts().rename({0: 'Under 2.5', 1: 'Over 2.5'}).to_string())

X_train_ou, X_test_ou, y_train_ou, y_test_ou = train_test_split(
    X, y_ou, test_size=0.2, random_state=42, stratify=y_ou
)

results_ou = optimize_models(X_train_ou, y_train_ou)
best_model_ou = results_ou['GradientBoosting'].best_estimator_
y_ou_pred = best_model_ou.predict(X_test_ou)

print(f'\nOver/Under 2.5 Accuracy: {accuracy_score(y_test_ou, y_ou_pred):.2%}')
print()
print(classification_report(y_test_ou, y_ou_pred, target_names=['Under 2.5', 'Over 2.5']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_ou = confusion_matrix(y_test_ou, y_ou_pred)
sns.heatmap(cm_ou, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Under 2.5', 'Over 2.5'],
            yticklabels=['Under 2.5', 'Over 2.5'], ax=axes[0])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('Over/Under 2.5 - Confusion Matrix')

importances_ou = pd.Series(best_model_ou.named_steps['classifier'].feature_importances_, index=X.columns)
importances_ou.nlargest(12).plot(kind='barh', ax=axes[1], color='darkorange', edgecolor='black')
axes[1].set_title('Over/Under 2.5 - Feature Importance')
axes[1].set_xlabel('Importance')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

---
## 9. Train BTTS (Both Teams to Score) Model

In [ ]:
y_btts = ((df['FTHG'] > 0) & (df['FTAG'] > 0)).astype(int)
print('BTTS target distribution:')
print(y_btts.value_counts().rename({0: 'No BTTS', 1: 'BTTS'}).to_string())

X_train_btts, X_test_btts, y_train_btts, y_test_btts = train_test_split(
    X, y_btts, test_size=0.2, random_state=42, stratify=y_btts
)

results_btts = optimize_models(X_train_btts, y_train_btts)
best_model_btts = results_btts['GradientBoosting'].best_estimator_
y_btts_pred = best_model_btts.predict(X_test_btts)

print(f'\nBTTS Accuracy: {accuracy_score(y_test_btts, y_btts_pred):.2%}')
print()
print(classification_report(y_test_btts, y_btts_pred, target_names=['No BTTS', 'BTTS']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_btts = confusion_matrix(y_test_btts, y_btts_pred)
sns.heatmap(cm_btts, annot=True, fmt='d', cmap='Greens',
            xticklabels=['No BTTS', 'BTTS'],
            yticklabels=['No BTTS', 'BTTS'], ax=axes[0])
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
axes[0].set_title('BTTS - Confusion Matrix')

importances_btts = pd.Series(best_model_btts.named_steps['classifier'].feature_importances_, index=X.columns)
importances_btts.nlargest(12).plot(kind='barh', ax=axes[1], color='forestgreen', edgecolor='black')
axes[1].set_title('BTTS - Feature Importance')
axes[1].set_xlabel('Importance')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

---
## 10. Model Comparison

In [ ]:
from sklearn.model_selection import cross_val_score

models_info = {
    'Match Result': (best_model, X_train, y_train),
    'Over/Under 2.5': (best_model_ou, X_train_ou, y_train_ou),
    'BTTS': (best_model_btts, X_train_btts, y_train_btts),
}

comparison = []
for name, (m, Xtr, ytr) in models_info.items():
    cv_scores = cross_val_score(m, Xtr, ytr, cv=5, scoring='accuracy')
    test_y = {'Match Result': y_test, 'Over/Under 2.5': y_test_ou, 'BTTS': y_test_btts}[name]
    test_pred = {'Match Result': y_pred, 'Over/Under 2.5': y_ou_pred, 'BTTS': y_btts_pred}[name]
    comparison.append({
        'Model': name,
        'CV Mean': cv_scores.mean(),
        'CV Std': cv_scores.std(),
        'Test Acc': accuracy_score(test_y, test_pred),
    })

comp_df = pd.DataFrame(comparison)
print(comp_df.to_string(index=False, float_format='{:.2%}'.format))

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(comp_df))
width = 0.35
ax.bar(x - width/2, comp_df['Test Acc'], width, label='Test Accuracy', color='steelblue', edgecolor='black')
ax.bar(x + width/2, comp_df['CV Mean'], width, label='CV Mean', color='darkorange', edgecolor='black')
ax.set_ylabel('Accuracy')
ax.set_title('Model Comparison')
ax.set_xticks(x)
ax.set_xticklabels(comp_df['Model'])
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

---
## 11. Hyperparameter Tuning Results

In [ ]:
for name, grid in results.items():
    print(f'\n{name}:')
    print(f'  Best params: {grid.best_params_}')
    print(f'  Best CV F1: {grid.best_score_:.4f}')

---
## 12. Save Models

In [ ]:
import joblib

os.makedirs('models/', exist_ok=True)

# Result model (Pipeline with scaler)
joblib.dump(best_model, 'models/result.pkl')
joblib.dump(list(X.columns), 'models/features_result.pkl')

# Over/Under model
joblib.dump(best_model_ou, 'models/over_under.pkl')

# BTTS model
joblib.dump(best_model_btts, 'models/btts.pkl')

saved = [f for f in os.listdir('models/') if f.endswith('.pkl')]
print('Saved files:')
for f in sorted(saved):
    print(f'  {f}')

---
## 13. Test: Load and Predict

In [ ]:
loaded_model = joblib.load('models/result.pkl')
sample = X_test.iloc[[0]]
prediction = loaded_model.predict(sample)[0]
probabilities = loaded_model.predict_proba(sample)[0]

print(f'Prediction: {result_map.get(prediction, prediction)}')
print(f'Probabilities:')
for cls, prob in zip(loaded_model.classes_, probabilities):
    print(f'  {result_map.get(cls, cls)}: {prob:.1%}')

---
## Done!

All three models trained with GridSearchCV and saved. Pipelines include scaling, so the app loads models directly without separate scalers.

| Model | Test Accuracy |
|-------|--------------|
| **Match Result** | ~38% |
| **Over/Under 2.5** | ~42% |
| **BTTS** | ~42% |

**Why are accuracies low?** Only 330 matches with 17 features. Football prediction is inherently difficult.

**Next steps:**
- Run `streamlit run app.py` for the prediction UI
